In [ ]:
import pandas as pd

# Paths to your CSVs
files = [
    "../data/Daily-Financial-News/analyst_ratings_processed.csv",
    "../data/Daily-Financial-News/raw_analyst_ratings.csv",
    "../data/Daily-Financial-News/raw_partner_headlines.csv",
    "../data/financial-phrase-bank/all-data.csv"
]

# Show first 5 rows of each file
for f in files:
    try:
        df = pd.read_csv(f)
        print(f"\n===== {f} =====")
        print(f"Shape: {df.shape}")
        print(df.head())
    except Exception as e:
        print(f"Error reading {f}: {e}")


In [80]:
# Cell 1: Imports & basic setup (torchtext-free)
import re
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, accuracy_score

# Device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)


Using device: cpu


In [81]:
# Cell 2: Load cleaned Financial Phrase Bank CSV
# Format: first column = label string, second = text
df = pd.read_csv("../data/financial-phrase-bank/all-data.csv",
                 encoding="latin-1", header=None, names=["label", "text"])

label_mapping = {"negative": 0, "neutral": 1, "positive": 2}
df["label"] = df["label"].map(label_mapping)

print("Dataset shape:", df.shape)
print(df.head())
print("Class distribution:\n", df["label"].value_counts())


Dataset shape: (4846, 2)
   label                                               text
0      1  According to Gran , the company has no plans t...
1      1  Technopolis plans to develop in stages an area...
2      0  The international electronic industry company ...
3      2  With the new production plant the company woul...
4      2  According to the company 's updated strategy f...
Class distribution:
 label
1    2879
2    1363
0     604
Name: count, dtype: int64


In [82]:
# Cell 3: Split
X_train, X_tmp, y_train, y_tmp = train_test_split(
    df["text"], df["label"], test_size=0.2, random_state=42, stratify=df["label"]
)
X_val, X_test, y_val, y_test = train_test_split(
    X_tmp, y_tmp, test_size=0.5, random_state=42, stratify=y_tmp
)

print(f"Train: {len(X_train)} | Val: {len(X_val)} | Test: {len(X_test)}")


Train: 3876 | Val: 485 | Test: 485


In [83]:
# Cell 4: Tokenizer + vocab (no torchtext)
def simple_tokenize(text):
    # lowercase, keep words and numbers
    return re.findall(r"[a-z0-9]+", str(text).lower())

from collections import Counter

MAX_VOCAB = 20000  # cap vocab size
PAD_ID = 0
UNK_ID = 1

def build_vocab(texts, max_vocab=MAX_VOCAB):
    counter = Counter()
    for t in texts:
        counter.update(simple_tokenize(t))
    most_common = counter.most_common(max_vocab - 2)  # reserve PAD, UNK
    stoi = {"<pad>": PAD_ID, "<unk>": UNK_ID}
    for i, (tok, _) in enumerate(most_common, start=2):
        stoi[tok] = i
    itos = {i: s for s, i in stoi.items()}
    return stoi, itos

stoi, itos = build_vocab(X_train)
print("Vocab size:", len(stoi))


Vocab size: 8998


In [84]:
# Cell 5: Dataset
class SentimentDataset(Dataset):
    def __init__(self, texts, labels, stoi, max_len=50):
        self.texts = texts.reset_index(drop=True)
        self.labels = labels.reset_index(drop=True)
        self.stoi = stoi
        self.max_len = max_len

    def encode(self, text):
        toks = simple_tokenize(text)
        ids = [self.stoi.get(t, UNK_ID) for t in toks][:self.max_len]
        if len(ids) < self.max_len:
            ids += [PAD_ID] * (self.max_len - len(ids))
        return ids

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        x = torch.tensor(self.encode(self.texts.iloc[idx]), dtype=torch.long)
        y = torch.tensor(self.labels.iloc[idx], dtype=torch.long)
        return x, y


In [85]:
# Cell 6: Dataloaders
BATCH_SIZE = 64
MAX_LEN = 50

train_ds = SentimentDataset(X_train, y_train, stoi, max_len=MAX_LEN)
val_ds   = SentimentDataset(X_val,   y_val,   stoi, max_len=MAX_LEN)
test_ds  = SentimentDataset(X_test,  y_test,  stoi, max_len=MAX_LEN)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE)
test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE)


In [86]:
# Cell 7: LSTM model
class SentimentLSTM(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim, output_dim, num_layers=2, dropout=0.3):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=PAD_ID)
        self.lstm = nn.LSTM(embed_dim, hidden_dim, num_layers=num_layers,
                            batch_first=True, dropout=dropout)
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(hidden_dim, output_dim)

    def forward(self, x):
        emb = self.embedding(x)               # (B, T, E)
        out, (h, c) = self.lstm(emb)          # h: (num_layers, B, H)
        last = h[-1]                           # (B, H)
        last = self.dropout(last)
        logits = self.fc(last)                 # (B, C)
        return logits

model = SentimentLSTM(
    vocab_size=len(stoi), embed_dim=100, hidden_dim=128, output_dim=3, num_layers=2, dropout=0.3
).to(device)

sum(p.numel() for p in model.parameters()), model


(1150043,
 SentimentLSTM(
   (embedding): Embedding(8998, 100, padding_idx=0)
   (lstm): LSTM(100, 128, num_layers=2, batch_first=True, dropout=0.3)
   (dropout): Dropout(p=0.3, inplace=False)
   (fc): Linear(in_features=128, out_features=3, bias=True)
 ))

In [87]:
# Cell 8: Train loop + early stopping on val loss
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

EPOCHS = 12
best_val_loss = float("inf")
patience = 3
pat = 0

for epoch in range(1, EPOCHS+1):
    model.train()
    tr_loss = 0.0
    for xb, yb in train_loader:
        xb, yb = xb.to(device), yb.to(device)
        optimizer.zero_grad()
        logits = model(xb)
        loss = criterion(logits, yb)
        loss.backward()
        optimizer.step()
        tr_loss += loss.item()

    model.eval()
    va_loss = 0.0
    with torch.no_grad():
        for xb, yb in val_loader:
            xb, yb = xb.to(device), yb.to(device)
            logits = model(xb)
            loss = criterion(logits, yb)
            va_loss += loss.item()

    tr_loss /= max(1, len(train_loader))
    va_loss /= max(1, len(val_loader))
    print(f"Epoch {epoch:02d} | train {tr_loss:.4f} | val {va_loss:.4f}")

    if va_loss < best_val_loss - 1e-4:
        best_val_loss = va_loss
        pat = 0
        torch.save(model.state_dict(), "best_phrasebank_lstm.pt")
    else:
        pat += 1
        if pat >= patience:
            print("Early stopping.")
            break

# load best
model.load_state_dict(torch.load("best_phrasebank_lstm.pt"))


Epoch 01 | train 0.9464 | val 0.9175
Epoch 02 | train 0.9035 | val 0.8544
Epoch 03 | train 0.8487 | val 0.8634
Epoch 04 | train 0.8165 | val 0.8326
Epoch 05 | train 0.7739 | val 0.7929
Epoch 06 | train 0.7113 | val 0.7914
Epoch 07 | train 0.6473 | val 0.8265
Epoch 08 | train 0.6047 | val 0.8464
Epoch 09 | train 0.5521 | val 0.8206
Early stopping.


<All keys matched successfully>

In [88]:
# Cell 9: Test evaluation
model.eval()
y_true, y_pred = [], []

with torch.no_grad():
    for xb, yb in test_loader:
        xb = xb.to(device)
        logits = model(xb)
        preds = torch.argmax(logits, dim=1).cpu().numpy()
        y_true.extend(yb.numpy())
        y_pred.extend(preds)

acc = accuracy_score(y_true, y_pred)
print("Test Accuracy:", f"{acc*100:.2f}%")
print(classification_report(y_true, y_pred, target_names=["Negative", "Neutral", "Positive"]))


Test Accuracy: 65.98%
              precision    recall  f1-score   support

    Negative       0.00      0.00      0.00        61
     Neutral       0.74      0.89      0.81       288
    Positive       0.47      0.47      0.47       136

    accuracy                           0.66       485
   macro avg       0.40      0.45      0.42       485
weighted avg       0.57      0.66      0.61       485

